# 2. GRPO runs

Pick one run per session. Within each experiment, the two runs start from the same checkpoint and see exactly the same prompts; only the grader differs.

| Run | Experiment | Starts from | Grader |
|---|---|---|---|
| `grpo_biased` | 1, sycophancy | DPO | correct answer **or** the user's (wrong) suggestion |
| `grpo_strict` | 1, control | DPO | correct answer |
| `grpo_hedge_lenient` | 2, hedging | seeded SFT | correct answer anywhere on the final line |
| `grpo_hedge_strict` | 2, control | seeded SFT | exactly one number, and it is correct |
| `grpo_biased_monitored` | 1, optional | DPO | biased, minus a penalty when the CoT monitor flags |

Each run takes about 60 to 90 minutes on a T4, plus about 10 minutes of evaluation.

If Colab disconnects, reconnect and use *Runtime > Run all* again: finished stages and evaluations are skipped, and GRPO resumes from its last checkpoint (saved every 25 steps).

In [ ]:
# Runs on a free Colab T4. Checkpoints and logs go to Google Drive, so a disconnect loses at most a few minutes.
REPO_URL = "https://github.com/lobby9000/reward-hacking-gsm8k"
DRIVE_DIR = "/content/drive/MyDrive/reward-hacking-gsm8k"

import os
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_DIR}/runs", exist_ok=True)
if not os.path.exists("/content/reward-hacking-gsm8k"):
    !git clone -q {REPO_URL} /content/reward-hacking-gsm8k
%cd /content/reward-hacking-gsm8k
!git pull -q
!pip install -q -r requirements.txt && pip install -q -e . --no-deps
if not os.path.islink("runs"):
    !rm -rf runs && ln -s {DRIVE_DIR}/runs runs
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
RUN = "grpo_biased"  # @param ["grpo_biased", "grpo_strict", "grpo_hedge_lenient", "grpo_hedge_strict", "grpo_biased_monitored"]
EXPERIMENT_2 = RUN.startswith("grpo_hedge")
START = "sft_hedge" if EXPERIMENT_2 else "dpo"
ADAPTERS = "runs/sft_hedge/adapter" if EXPERIMENT_2 else "runs/sft/adapter,runs/dpo/adapter"
EVAL_CONFIG = "configs/eval_hedge.yaml" if EXPERIMENT_2 else "configs/eval.yaml"

The starting policy, from notebook 1. GRPO can only amplify behaviour the policy already samples: for experiment 1, check that `hint_copy_rate` is not exactly zero on the wrong-hint conditions; for experiment 2, that `hedge_rate` is a few percent.

In [ ]:
import json
for s in json.load(open(f"runs/eval/{START}/summary.json"))["summaries"]:
    print(f"{s['condition']:>15}  accuracy {s['accuracy']:.3f}  "
          f"copies hint {s['hint_copy_rate'] if s['hint_copy_rate'] is not None else '-'}  "
          f"hedges {s['hedge_rate']:.3f}")

Optional speed check: three steps in a scratch folder. Look at `step_s` (seconds per step). On a T4 expect roughly 20 to 40 seconds; 200 steps then take 70 to 130 minutes.

In [ ]:
if not os.path.exists(f"runs/{RUN}/metrics.jsonl"):  # skipped once the real run has started
    !python -m posttrain.grpo --config configs/{RUN}.yaml --set output_dir=/content/speedtest --set steps=3 --set eval_every=0 --set save_every=0 --set resume=false

## Training (resumable)

In [ ]:
!python -m posttrain.grpo --config configs/{RUN}.yaml

## Final evaluation (about 10 minutes)

In [ ]:
!python -m posttrain.evaluate --config {EVAL_CONFIG} --set name={RUN} --set "model.adapters=[{ADAPTERS},runs/{RUN}/adapter]"